# 02 — Train a DQN Model Offline

Load the transitions collected by `01_collect_dataset.ipynb` and train a small DQN-style MOUSE model entirely from stored data — no live environment needed.

The model has three independently-configurable pieces:

| Piece | Role |
|---|---|
| `StepEmbedder` | Encodes each `(action, observation, reward, done)` step into a sequence of vectors |
| `Qwen3Backbone` | A small transformer that reads those vectors and builds up context over time |
| `DiscreteActionValueHead` | Predicts a Q-value for each possible action |

`Qwen3Backbone` downloads a public pretrained checkpoint on first run and exposes `hidden_dim`, which sizes everything else.

In [1]:
import torch

from mouse_core.data import DataLoader, load_stores_from_hub
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead

# Hub dataset repo written by 01_collect_dataset.ipynb.
DATASET_ID = "mouse-example-dataset"

# Hub model repo where the trained checkpoint is uploaded.
MODEL_ID = "mouse-example-model"

# FrozenLake action count; used for the action embedding and DQN head width.
MAX_ACTIONS = 4

# FrozenLake 4x4 has 16 discrete observation states.
MAX_OBS_DISCRETE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load data

`load_stores_from_hub` downloads each named environment config from the Hub. The `DataLoader` accepts the list of stores directly and mixes them during training — no manual merge needed.

In [2]:
stores = load_stores_from_hub(DATASET_ID, split="train")
for s in stores:
    print(s)

Datastore(name='frozenlake_deterministic#0', steps=200)
Datastore(name='frozenlake_slippery#0', steps=200)
Datastore(name='frozenlake_slippery#1', steps=200)


## DataLoader

`DataLoader` slices one or more `Datastore`s into fixed-length sequence batches. Multiple stores are mixed automatically. Call `loader.next_batch()` to get a `list[list[dict]]` of shape `[B][S]` — one inner list per sequence in the batch, one dict per step. `model.forward` accepts this directly and handles encoding internally.

In [3]:
loader = DataLoader(stores, sequence_length=8, batch_size=4, prefetch=2, num_workers=0)

## Build the model

`Qwen3Backbone` downloads `Qwen/Qwen3-0.6B` on first run and keeps only the first `num_layers` transformer layers. `StepEmbedder` embeds each step into tokens, the backbone mixes them over time, and `DiscreteActionValueHead` maps the result to per-action Q-values — all sized by `backbone.hidden_dim`.

In [4]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B", num_layers=2)

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {"name": "action",      "embed": "discrete",   "vocab_size": MAX_ACTIONS,     "std": 0.02, "tokens": 1},
        {"name": "observation", "embed": "discrete",   "vocab_size": MAX_OBS_DISCRETE, "std": 0.02, "tokens": 1},
        {"name": "reward",      "embed": "rff",        "std": 0.02, "in_min": 0.01, "in_max": 10.0, "tokens": 1},
        {"name": "done",        "embed": "discrete",   "vocab_size": 5,               "std": 0.02, "tokens": 1},
        {"name": "scratch",     "embed": "learnable",  "tokens": 1},
    ],
    concat_modalities=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.01,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(16, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
      (scratch): LearnableEmbedder()
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-1): 2 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linea

## Train

Each step:
1. Sample a random batch of transition sequences from the store.
2. Run the model forward to get Q-value predictions.
3. Compute the Bellman loss — `objective(objective_data, predictions)` returns `(loss, metrics)`.
4. Update the model with gradient descent.
5. Slowly copy weights to a frozen target network (Polyak update) for stable targets.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
objective = DqnObjective(gamma=0.99, tau=0.005)


def train_step(batch):
    predictions, objective_data, _ = model(batch)
    loss, _ = objective(objective_data, predictions)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    model.polyak_update(action_value_tau=objective.tau)
    return loss.item()


for step in range(200):
    loss = train_step(loader.next_batch())
    if step % 10 == 0:
        print(f"step={step:3d}  loss={loss:.4f}")

loader.close()
print("Training finished.")

step=  0  loss=0.0001
step= 10  loss=0.0007
step= 20  loss=0.0001
step= 30  loss=0.0011
step= 40  loss=0.0012
step= 50  loss=0.0054
step= 60  loss=0.0005
step= 70  loss=0.0001
step= 80  loss=0.0001
step= 90  loss=0.0007
step=100  loss=0.0005
step=110  loss=0.0023
step=120  loss=0.0007
step=130  loss=0.0002
step=140  loss=0.0002
step=150  loss=0.0306
step=160  loss=0.0007
step=170  loss=0.0004
step=180  loss=0.0517
step=190  loss=0.0006
Training finished.


## Push to the Hub

Upload the trained checkpoint. The next notebook loads it by the same `MODEL_ID` to run inference.

In [6]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")

Pushed to https://huggingface.co/micahr234/mouse-example-model
